## Setup

In [1]:
import polars as pl

from tdt4225_ex2.eda import get_data

pl.Config(set_tbl_cols=1000)

df = get_data(verify=True)

## Duplicated TRIP_IDs

In [2]:
dupe_counts = (
    df.group_by("TRIP_ID").agg(pl.len().alias("count")).filter(pl.col("count") > 1)
)
print(dupe_counts)

dupe_ids = dupe_counts["TRIP_ID"].implode()

all_dupes = df.filter(pl.col("TRIP_ID").is_in(dupe_ids)).sort(["TRIP_ID", "TIMESTAMP"])
print(all_dupes)

shape: (80, 2)
┌─────────────────────┬───────┐
│ TRIP_ID             ┆ count │
│ ---                 ┆ ---   │
│ i64                 ┆ u32   │
╞═════════════════════╪═══════╡
│ 1397199117620000513 ┆ 2     │
│ 1381645412620000328 ┆ 2     │
│ 1392640481620000663 ┆ 2     │
│ 1385928574620000080 ┆ 2     │
│ 1398185157620000066 ┆ 2     │
│ …                   ┆ …     │
│ 1389782974620000562 ┆ 2     │
│ 1401469148620000104 ┆ 2     │
│ 1399918717620000571 ┆ 2     │
│ 1392317061620000066 ┆ 2     │
│ 1383557098620000562 ┆ 2     │
└─────────────────────┴───────┘
shape: (161, 9)
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ TRIP_ID  ┆ CALL_TYP ┆ ORIGIN_C ┆ ORIGIN_S ┆ TAXI_ID  ┆ TIMESTAM ┆ DAY_TYPE ┆ MISSING_ ┆ POLYLINE │
│ ---      ┆ E        ┆ ALL      ┆ TAND     ┆ ---      ┆ P        ┆ ---      ┆ DATA     ┆ ---      │
│ i64      ┆ ---      ┆ ---      ┆ ---      ┆ i64      ┆ ---      ┆ str      ┆ ---      ┆ list[lis │
│          ┆ str     

## DAY_TYPE distribution, overall and per date

In [3]:
print(df["DAY_TYPE"].value_counts(sort=True))
day_types_per_date = (
    df.group_by(pl.col("TIMESTAMP").dt.date().alias("DATE"))
    .agg(pl.col("DAY_TYPE").unique().sort().alias("DAY_TYPES"))
    .sort("DATE")
)
print(day_types_per_date.filter(pl.col("DAY_TYPES").list.len() > 1))

shape: (1, 2)
┌──────────┬─────────┐
│ DAY_TYPE ┆ count   │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ A        ┆ 1710670 │
└──────────┴─────────┘
shape: (0, 2)
┌──────┬───────────┐
│ DATE ┆ DAY_TYPES │
│ ---  ┆ ---       │
│ date ┆ list[str] │
╞══════╪═══════════╡
└──────┴───────────┘


## MISSING_DATA distribution, overall and per trip

In [4]:
print(df["MISSING_DATA"].value_counts(sort=True))
print(
    df.group_by("MISSING_DATA").agg(
        trips=pl.len(),
        empty=(pl.col("POLYLINE").list.len() == 0).sum(),
        under_3=(pl.col("POLYLINE").list.len() < 3).sum(),
        avg_points=pl.col("POLYLINE").list.len().mean(),
    )
)

shape: (2, 2)
┌──────────────┬─────────┐
│ MISSING_DATA ┆ count   │
│ ---          ┆ ---     │
│ bool         ┆ u32     │
╞══════════════╪═════════╡
│ false        ┆ 1710660 │
│ true         ┆ 10      │
└──────────────┴─────────┘
shape: (2, 5)
┌──────────────┬─────────┬───────┬─────────┬────────────┐
│ MISSING_DATA ┆ trips   ┆ empty ┆ under_3 ┆ avg_points │
│ ---          ┆ ---     ┆ ---   ┆ ---     ┆ ---        │
│ bool         ┆ u32     ┆ u32   ┆ u32     ┆ f64        │
╞══════════════╪═════════╪═══════╪═════════╪════════════╡
│ false        ┆ 1710660 ┆ 5901  ┆ 43902   ┆ 48.758033  │
│ true         ┆ 10      ┆ 0     ┆ 2       ┆ 96.9       │
└──────────────┴─────────┴───────┴─────────┴────────────┘


## Visualize trip paths

In [5]:
import random
from typing import Literal, TypedDict

import pydeck as pdk


class TripPath(TypedDict):
    path: list[list[float]]
    trip_id: int
    color: int


def get_color(seed: int) -> tuple[int, int, int, Literal[128]]:
    random.seed(seed)
    return tuple(random.randrange(60, 256) for _ in range(3)) + (128,)


def visualize_trip_paths(data: list[TripPath]) -> None:
    layer = pdk.Layer(
        "PathLayer",
        data,
        get_path="path",
        get_color="color",
        # get_color=[255, 128, 0, 128],
        width_min_pixels=1,
        pickable=True,
    )
    view = pdk.ViewState(latitude=41.17, longitude=-8.65, zoom=11)
    pdk.Deck(
        layers=[layer],
        initial_view_state=view,
        map_style="dark",
        tooltip={"text": "Trip {trip_id}"},
    ).to_html("routes.html")


data: list[TripPath] = [
    TripPath(
        path=row["POLYLINE"],
        trip_id=row["TRIP_ID"],
        color=get_color(row["TAXI_ID"]),
    )
    for row in df.sample(100000).iter_rows(named=True)
]
visualize_trip_paths(data)